In [2]:
from google import genai
from dotenv import load_dotenv

load_dotenv()
client = genai.Client()   # reads GOOGLE_API_KEY from the env
for m in client.models.list():
    print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.6-preview
models/gem

In [3]:
import index

store = index.build_index()
for d in store.similarity_search("What was Apple's gross profit in fiscal 2025?", k=3):
    print(d.metadata["id"])
    print(d.page_content[:300])
    print("---")

/Users/snehagampa/Documents/Varun/portfolio/ai_projects/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5001.24it/s]


indexed 144 chunks (paragraphs)
AAPL-Item 7-5
Services gross margin increased during 2025 compared to 2024 primarily due to higher Services net sales and a different mix of services.

Services gross margin percentage increased during 2025 compared to 2024 primarily due to a different mix of services, partially offset by higher costs.

The Compa
---
AAPL-Item 8-11
As of September 27, 2025 and September 28, 2024, the Company had total deferred revenue of $13.7 billion and $12.8 billion, respectively. As of September 27, 2025, the Company expects 66% of total deferred revenue to be realized in less than a year, 23% within one-to-two years, 9% within two-to-thre
---
AAPL-Item 8-17
The following table shows the Company’s gross property, plant and equipment by major asset class and accumulated depreciation as of September 27, 2025 and September 28, 2024 (in millions):

20252024

Land and buildings$27,337 $24,690 

Machinery, equipment and internal-use software

83,420 80,205 


---


In [4]:
# 1. Does any chunk contain the number?
docs = store.similarity_search("gross margin", k=5)
for d in docs:
    print(d.metadata["id"], "195,201" in d.page_content)

# 2. How deep is it for the ORIGINAL question?
docs = store.similarity_search("What was Apple's gross profit in fiscal 2025?", k=10)
for i, d in enumerate(docs):
    print(i, d.metadata["id"], "195,201" in d.page_content)

AAPL-Item 7-5 False
AAPL-Item 1A-41 False
AAPL-Item 8-15 False
AAPL-Item 8-3 False
AAPL-Item 8-14 False
0 AAPL-Item 7-5 False
1 AAPL-Item 8-11 False
2 AAPL-Item 8-17 False
3 AAPL-Item 7-4 True
4 AAPL-Item 7-3 False
5 AAPL-Item 8-1 False
6 AAPL-Item 8-19 False
7 AAPL-Item 8-20 False
8 AAPL-Item 5-1 False
9 AAPL-Item 7-0 False


In [5]:
import index
store = index.build_index()
docs = store.similarity_search("What was Apple's cost of sales in fiscal 2025?", k=10)
for i, d in enumerate(docs):
    print(i, d.metadata["id"], "220,960" in d.page_content)
    

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10801.11it/s]


indexed 144 chunks (paragraphs)
0 AAPL-Item 7-5 False
1 AAPL-Item 7-4 False
2 AAPL-Item 7-3 False
3 AAPL-Item 7-0 False
4 AAPL-Item 8-11 False
5 AAPL-Item 5-1 False
6 AAPL-Item 6-0 False
7 AAPL-Item 8-17 False
8 AAPL-Item 8-19 False
9 AAPL-Item 8-2 False


In [6]:
import json
from chunking import chunk_sections

# 1. Is it in the raw ingested text?
data = json.load(open("10k_aapl.json"))
for sec, text in data["sections"].items():
    if "220,960" in text:
        print("in source:", sec)

# 2. Did it survive chunking?
chunks = chunk_sections(data)
print("in chunks:", [c["id"] for c in chunks if "220,960" in c["text"]])

in source: Item 8
in chunks: ['AAPL-Item 8-0', 'AAPL-Item 8-30']


In [7]:
docs = store.similarity_search("What was Apple's cost of sales in fiscal 2025?", k=144)
for i, d in enumerate(docs):
    if "220,960" in d.page_content:
        print("rank:", i, d.metadata["id"])

rank: 15 AAPL-Item 8-0
rank: 82 AAPL-Item 8-30


In [18]:
import retrieval_validation
import retrieval
import evals

question = "What was Apple's total net sales in fiscal 2025?"
answer, docs = retrieval.ask(question)

answers_numbers = evals.extract_numbers(answer)
print(answers_numbers)
docs_numbers = []
for doc in docs:
        docs_numbers.extend(evals.extract_numbers(doc.page_content))
        print(docs_numbers)
docs_numbers = list(map(int, docs_numbers))
for i in answers_numbers:
    if int(answers_numbers[i]) in docs_numbers:
        True 
    else:
        False
    # 1. numbers in the answer  = extract_numbers(answer)
    # 2. numbers in the chunks  = extract_numbers of each doc's page_content, combined
    # 3. return True only if EVERY answer number is close to SOME chunk number
    #    (reuse similar_numbers — "$416,161 million" in the answer is
    #     "416,161" in the chunk, so compare scaled values with tolerance)

[2025.0, 416161000000.0, 2.0]
[1.0, 2025.0, 2024.0, 2025.0, 2024.0, 2025.0, 2024.0, 2025.0, 2024.0, 2025.0, 2024.0, 2025.0, 10.0, 23.0, 2025.0, 2024.0, 2023.0, 202520242023.0, 112887.0, 109633.0, 108803.0, 82314.0, 71050.0, 60345.0, 195201.0, 180683.0, 169148.0, 36.8, 37.2, 36.5, 75.4, 73.9, 70.8, 46.9, 46.2, 44.1, 2025.0, 2024.0, 2025.0, 2024.0]
[1.0, 2025.0, 2024.0, 2025.0, 2024.0, 2025.0, 2024.0, 2025.0, 2024.0, 2025.0, 2024.0, 2025.0, 10.0, 23.0, 2025.0, 2024.0, 2023.0, 202520242023.0, 112887.0, 109633.0, 108803.0, 82314.0, 71050.0, 60345.0, 195201.0, 180683.0, 169148.0, 36.8, 37.2, 36.5, 75.4, 73.9, 70.8, 46.9, 46.2, 44.1, 2025.0, 2024.0, 2025.0, 2024.0, 2025.0, 2024.0, 2023.0, 2025.0, 2024.0, 2023.0, 178353.0, 7.0, 167045.0, 3.0, 162560.0, 111032.0, 10.0, 101328.0, 7.0, 94294.0, 64377.0, 4.0, 66952.0, 8.0, 72559.0, 28703.0, 15.0, 25052.0, 3.0, 24257.0, 33696.0, 10.0, 30658.0, 4.0, 29615.0, 416161.0, 6.0, 391035.0, 2.0, 383285.0, 2025.0, 2024.0, 2025.0, 2025.0, 2024.0, 2025.0, 202

TypeError: list indices must be integers or slices, not float